# 02 EDA: KPI, Funnel, and Segment Analysis

This notebook performs exploratory analysis on processed tables to establish baseline KPIs, diagnose funnel drop-offs, compare segment behavior, and prepare hypotheses for A/B testing and uplift modeling.

**Expected processed inputs**
- `event_master.csv`
- `purchase_master.csv`
- `session_level_funnel.csv`
- `customer_level_features.csv`


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

BASE_PATH = "../data/processed"

print("Loading processed tables...")

event_master = pd.read_csv(os.path.join(BASE_PATH, "event_master.csv"))
purchase_master = pd.read_csv(os.path.join(BASE_PATH, "purchase_master.csv"))
session_df = pd.read_csv(os.path.join(BASE_PATH, "session_level_funnel.csv"))
customer_df = pd.read_csv(os.path.join(BASE_PATH, "customer_level_features.csv"))

date_cols_event = ["timestamp", "signup_date", "launch_date", "start_date", "end_date"]
for col in date_cols_event:
    if col in event_master.columns:
        event_master[col] = pd.to_datetime(event_master[col], errors="coerce")

if "timestamp" in purchase_master.columns:
    purchase_master["timestamp"] = pd.to_datetime(purchase_master["timestamp"], errors="coerce")

for col in ["session_start_ts", "session_end_ts"]:
    if col in session_df.columns:
        session_df[col] = pd.to_datetime(session_df[col], errors="coerce")

numeric_cols_pm = ["matched_tx_flag", "net_revenue", "gross_revenue_filled", "refund_flag_filled"]
for col in numeric_cols_pm:
    if col in purchase_master.columns:
        purchase_master[col] = pd.to_numeric(purchase_master[col], errors="coerce").fillna(0)

numeric_cols_sdf = [
    "any_view", "any_click", "any_add_to_cart", "any_purchase", "is_converted",
    "avg_session_duration_sec", "max_session_duration_sec", "wallclock_session_duration_sec"
]
for col in numeric_cols_sdf:
    if col in session_df.columns:
        session_df[col] = pd.to_numeric(session_df[col], errors="coerce").fillna(0)

numeric_cols_cdf = [
    "total_sessions", "converted_sessions", "total_orders", "net_revenue_total",
    "gross_revenue_total", "refunded_orders", "total_add_to_cart", "premium_order_share",
    "discount_sensitivity_score", "has_purchase_history"
]
for col in numeric_cols_cdf:
    if col in customer_df.columns:
        customer_df[col] = pd.to_numeric(customer_df[col], errors="coerce").fillna(0)

print("Loaded successfully")
print(f"event_master    : {event_master.shape}")
print(f"purchase_master : {purchase_master.shape}")
print(f"session_df      : {session_df.shape}")
print(f"customer_df     : {customer_df.shape}")


In [ ]:
data_frames = {
    "event_master": event_master,
    "purchase_master": purchase_master,
    "session_df": session_df,
    "customer_df": customer_df
}

for name, df in data_frames.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("shape:", df.shape)
    print("columns:")
    print(df.columns.tolist())
    display(df.head(5))


In [ ]:
# ================================================================
# STEP 1. Sanity Check
# ================================================================

em = event_master.copy()
sdf = session_df.copy()

for col in ["signup_to_event_days", "product_age_days"]:
    if col in em.columns:
        em[col] = pd.to_numeric(em[col], errors="coerce")

declared_candidates = [c for c in ["max_session_duration_sec", "avg_session_duration_sec", "wallclock_session_duration_sec"] if c in sdf.columns]
if len(declared_candidates) == 0:
    raise ValueError("No session duration column found in session_df")
declared_col = declared_candidates[0]
declared_duration = pd.to_numeric(sdf[declared_col], errors="coerce")
wallclock_duration = pd.to_numeric(sdf["wallclock_session_duration_sec"], errors="coerce") if "wallclock_session_duration_sec" in sdf.columns else pd.Series(index=sdf.index, dtype=float)

neg_signup_mask = em["signup_to_event_days"] < 0 if "signup_to_event_days" in em.columns else pd.Series(False, index=em.index)
valid_product_age_mask = em["product_age_days"].notna() if "product_age_days" in em.columns else pd.Series(False, index=em.index)
neg_product_age_mask = valid_product_age_mask & (em["product_age_days"] < 0) if "product_age_days" in em.columns else pd.Series(False, index=em.index)

p995_declared = declared_duration.quantile(0.995)
neg_declared_mask = declared_duration < 0
zero_declared_mask = declared_duration == 0
long_declared_mask = declared_duration > p995_declared

if wallclock_duration.notna().sum() > 0:
    p995_wallclock = wallclock_duration.quantile(0.995)
    neg_wallclock_mask = wallclock_duration < 0
    long_wallclock_mask = wallclock_duration > p995_wallclock
    mismatch_mask = declared_duration.notna() & wallclock_duration.notna() & (np.abs(declared_duration - wallclock_duration) > 600)
else:
    neg_wallclock_mask = pd.Series(False, index=sdf.index)
    long_wallclock_mask = pd.Series(False, index=sdf.index)
    mismatch_mask = pd.Series(False, index=sdf.index)

summary = pd.DataFrame({
    "metric": [
        "events rows",
        "sessions rows",
        "signup_to_event_days < 0",
        "product_age_days < 0 (valid only)",
        f"{declared_col} < 0",
        f"{declared_col} = 0",
        f"{declared_col} > p99.5",
        "wallclock_session_duration_sec < 0",
        "wallclock_session_duration_sec > p99.5",
        "declared vs wallclock mismatch (>600s)"
    ],
    "count": [
        len(em),
        len(sdf),
        int(neg_signup_mask.sum()),
        int(neg_product_age_mask.sum()),
        int(neg_declared_mask.sum()),
        int(zero_declared_mask.sum()),
        int(long_declared_mask.sum()),
        int(neg_wallclock_mask.sum()),
        int(long_wallclock_mask.sum()),
        int(mismatch_mask.sum())
    ],
    "ratio": [
        1.0,
        1.0,
        neg_signup_mask.mean() if len(em) > 0 else np.nan,
        neg_product_age_mask.mean() if valid_product_age_mask.sum() > 0 else np.nan,
        neg_declared_mask.mean() if len(sdf) > 0 else np.nan,
        zero_declared_mask.mean() if len(sdf) > 0 else np.nan,
        long_declared_mask.mean() if len(sdf) > 0 else np.nan,
        neg_wallclock_mask.mean() if len(sdf) > 0 else np.nan,
        long_wallclock_mask.mean() if len(sdf) > 0 else np.nan,
        mismatch_mask.mean() if len(sdf) > 0 else np.nan
    ]
})

print("\n==================== Core sanity-check summary ====================")
display(summary)

event_count_col = "event_id" if "event_id" in em.columns else None

signup_seg_cols = [c for c in ["country", "loyalty_tier", "acquisition_channel", "traffic_source", "device_type"] if c in em.columns]
for col in signup_seg_cols:
    if event_count_col is not None:
        tmp = em.groupby(col, dropna=False).agg(total_events=(event_count_col, "count"), neg_signup_events=("signup_to_event_days", lambda s: (pd.to_numeric(s, errors="coerce") < 0).sum())).reset_index()
    else:
        tmp = em.groupby(col, dropna=False).size().reset_index(name="total_events")
        neg_tmp = em.groupby(col, dropna=False)["signup_to_event_days"].apply(lambda s: (pd.to_numeric(s, errors="coerce") < 0).sum()).reset_index(name="neg_signup_events")
        tmp = tmp.merge(neg_tmp, on=col, how="left")
    tmp["neg_signup_rate"] = np.where(tmp["total_events"] > 0, tmp["neg_signup_events"] / tmp["total_events"], np.nan)
    print(f"\nTop segments for negative signup_to_event_days by {col}")
    display(tmp.sort_values(["neg_signup_rate", "neg_signup_events"], ascending=[False, False]).head(10))

prod_seg_cols = [c for c in ["category", "is_premium", "traffic_source", "page_category", "event_type"] if c in em.columns]
base_prod = em.loc[em["product_age_days"].notna()].copy() if "product_age_days" in em.columns else em.head(0).copy()
for col in prod_seg_cols:
    if len(base_prod) == 0:
        continue
    if event_count_col is not None:
        tmp = base_prod.groupby(col, dropna=False).agg(total_valid_events=(event_count_col, "count"), neg_product_age_events=("product_age_days", lambda s: (pd.to_numeric(s, errors="coerce") < 0).sum())).reset_index()
    else:
        tmp = base_prod.groupby(col, dropna=False).size().reset_index(name="total_valid_events")
        neg_tmp = base_prod.groupby(col, dropna=False)["product_age_days"].apply(lambda s: (pd.to_numeric(s, errors="coerce") < 0).sum()).reset_index(name="neg_product_age_events")
        tmp = tmp.merge(neg_tmp, on=col, how="left")
    tmp["neg_product_age_rate"] = np.where(tmp["total_valid_events"] > 0, tmp["neg_product_age_events"] / tmp["total_valid_events"], np.nan)
    print(f"\nTop segments for negative product_age_days by {col}")
    display(tmp.sort_values(["neg_product_age_rate", "neg_product_age_events"], ascending=[False, False]).head(10))

duration_summary = pd.DataFrame({
    "metric": [
        f"{declared_col} min", f"{declared_col} p1", f"{declared_col} p50", f"{declared_col} p95", f"{declared_col} p99", f"{declared_col} p99.5", f"{declared_col} max",
        "wallclock min", "wallclock p1", "wallclock p50", "wallclock p95", "wallclock p99", "wallclock p99.5", "wallclock max"
    ],
    "value": [
        declared_duration.min(),
        declared_duration.quantile(0.01),
        declared_duration.quantile(0.50),
        declared_duration.quantile(0.95),
        declared_duration.quantile(0.99),
        declared_duration.quantile(0.995),
        declared_duration.max(),
        wallclock_duration.min() if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.01) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.50) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.95) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.99) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.quantile(0.995) if wallclock_duration.notna().sum() > 0 else np.nan,
        wallclock_duration.max() if wallclock_duration.notna().sum() > 0 else np.nan
    ]
})
print("\n==================== Session duration distribution summary ====================")
display(duration_summary)

sdf["duration_outlier_flag"] = (neg_declared_mask | zero_declared_mask | long_declared_mask | neg_wallclock_mask | long_wallclock_mask | mismatch_mask).astype(int)
session_seg_cols = [c for c in ["first_device_type", "first_traffic_source", "first_page_category", "experiment_group", "funnel_stage"] if c in sdf.columns]
for col in session_seg_cols:
    tmp = sdf.groupby(col, dropna=False).agg(total_sessions=("session_id", "count"), duration_outlier_sessions=("duration_outlier_flag", "sum")).reset_index()
    tmp["duration_outlier_rate"] = np.where(tmp["total_sessions"] > 0, tmp["duration_outlier_sessions"] / tmp["total_sessions"], np.nan)
    print(f"\nTop session-duration outlier segments by {col}")
    display(tmp.sort_values(["duration_outlier_rate", "duration_outlier_sessions"], ascending=[False, False]).head(10))


In [ ]:
# ================================================================
# STEP 2. KPI baseline and overall funnel
# ================================================================

em = event_master.copy()
pm = purchase_master.copy()
sdf = session_df.copy()
cdf = customer_df.copy()

valid_orders = pm[pm["matched_tx_flag"] == 1].copy() if "matched_tx_flag" in pm.columns else pm.copy()

view_sessions = sdf["any_view"].sum() if "any_view" in sdf.columns else np.nan
click_sessions = sdf["any_click"].sum() if "any_click" in sdf.columns else np.nan
cart_sessions = sdf["any_add_to_cart"].sum() if "any_add_to_cart" in sdf.columns else np.nan
purchase_sessions = sdf["any_purchase"].sum() if "any_purchase" in sdf.columns else np.nan

base_kpi = pd.DataFrame({
    "metric": [
        "Total customers",
        "Total sessions",
        "Total purchases (matched transactions)",
        "Session conversion rate",
        "Average sessions per customer",
        "Average orders per customer",
        "Gross revenue",
        "Net revenue",
        "Average order value (AOV)",
        "Refund rate",
        "Overall view->click rate",
        "Overall click->cart rate",
        "Overall cart->purchase rate"
    ],
    "value": [
        cdf["customer_id"].nunique() if "customer_id" in cdf.columns else np.nan,
        len(sdf),
        len(valid_orders),
        sdf["is_converted"].mean() if "is_converted" in sdf.columns else np.nan,
        cdf["total_sessions"].mean() if "total_sessions" in cdf.columns else np.nan,
        cdf["total_orders"].mean() if "total_orders" in cdf.columns else np.nan,
        valid_orders["gross_revenue_filled"].sum() if "gross_revenue_filled" in valid_orders.columns else np.nan,
        valid_orders["net_revenue"].sum() if "net_revenue" in valid_orders.columns else np.nan,
        (valid_orders["net_revenue"].sum() / len(valid_orders)) if ("net_revenue" in valid_orders.columns and len(valid_orders) > 0) else np.nan,
        valid_orders["refund_flag_filled"].mean() if "refund_flag_filled" in valid_orders.columns else np.nan,
        click_sessions / view_sessions if pd.notna(view_sessions) and view_sessions > 0 else np.nan,
        cart_sessions / click_sessions if pd.notna(click_sessions) and click_sessions > 0 else np.nan,
        purchase_sessions / cart_sessions if pd.notna(cart_sessions) and cart_sessions > 0 else np.nan
    ]
})

print("\n==================== KPI Baseline ====================")
display(base_kpi)

funnel_df = pd.DataFrame({
    "stage": ["view", "click", "add_to_cart", "purchase"],
    "sessions": [view_sessions, click_sessions, cart_sessions, purchase_sessions]
})
funnel_df["rate_from_prev"] = funnel_df["sessions"] / funnel_df["sessions"].shift(1)
funnel_df.loc[0, "rate_from_prev"] = 1.0

print("\n==================== Overall Funnel ====================")
display(funnel_df)

plt.figure(figsize=(8, 5))
plt.bar(funnel_df["stage"], funnel_df["sessions"])
plt.title("Overall Session Funnel")
plt.xlabel("Stage")
plt.ylabel("Number of Sessions")
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(funnel_df["stage"], funnel_df["rate_from_prev"], marker="o")
plt.title("Stage-to-Stage Conversion Rate")
plt.xlabel("Stage")
plt.ylabel("Conversion Rate")
plt.ylim(0, 1.05)
plt.show()

def funnel_by_segment(df, group_col):
    out = (
        df.groupby(group_col, dropna=False)
        .agg(
            total_sessions=("session_id", "count"),
            view_sessions=("any_view", "sum"),
            click_sessions=("any_click", "sum"),
            cart_sessions=("any_add_to_cart", "sum"),
            purchase_sessions=("any_purchase", "sum")
        )
        .reset_index()
    )
    out["session_conversion_rate"] = np.where(out["total_sessions"] > 0, out["purchase_sessions"] / out["total_sessions"], np.nan)
    out["view_to_click_rate"] = np.where(out["view_sessions"] > 0, out["click_sessions"] / out["view_sessions"], np.nan)
    out["click_to_cart_rate"] = np.where(out["click_sessions"] > 0, out["cart_sessions"] / out["click_sessions"], np.nan)
    out["cart_to_purchase_rate"] = np.where(out["cart_sessions"] > 0, out["purchase_sessions"] / out["cart_sessions"], np.nan)
    return out.sort_values("total_sessions", ascending=False)

if "first_device_type" in sdf.columns:
    device_funnel = funnel_by_segment(sdf, "first_device_type")
    print("\n==================== Funnel by Device ====================")
    display(device_funnel)

    plt.figure(figsize=(8, 5))
    plt.bar(device_funnel["first_device_type"].astype(str), device_funnel["session_conversion_rate"])
    plt.title("Session Conversion Rate by Device")
    plt.xlabel("Device")
    plt.ylabel("Session Conversion Rate")
    plt.show()

if "first_traffic_source" in sdf.columns:
    traffic_funnel = funnel_by_segment(sdf, "first_traffic_source")
    print("\n==================== Funnel by Traffic Source ====================")
    display(traffic_funnel)

    plt.figure(figsize=(8, 5))
    plt.bar(traffic_funnel["first_traffic_source"].astype(str), traffic_funnel["session_conversion_rate"])
    plt.title("Session Conversion Rate by Traffic Source")
    plt.xlabel("Traffic Source")
    plt.ylabel("Session Conversion Rate")
    plt.show()


In [ ]:
# ================================================================
# STEP 3. Customer / loyalty / problem segment diagnostics
# ================================================================

cdf2 = customer_df.copy()

cdf2["customer_status"] = np.where(cdf2["has_purchase_history"] == 1, "Existing_or_Buyer", "No_Purchase_History") if "has_purchase_history" in cdf2.columns else "Unknown"

status_summary = (
    cdf2.groupby("customer_status", dropna=False)
    .agg(
        total_customers=("customer_id", "count"),
        total_sessions=("total_sessions", "sum"),
        converted_sessions=("converted_sessions", "sum"),
        total_orders=("total_orders", "sum"),
        net_revenue_total=("net_revenue_total", "sum"),
        avg_sessions_per_customer=("total_sessions", "mean"),
        avg_orders_per_customer=("total_orders", "mean"),
        avg_revenue_per_customer=("net_revenue_total", "mean")
    )
    .reset_index()
)
status_summary["session_conversion_rate"] = np.where(status_summary["total_sessions"] > 0, status_summary["converted_sessions"] / status_summary["total_sessions"], np.nan)
status_summary["AOV"] = np.where(status_summary["total_orders"] > 0, status_summary["net_revenue_total"] / status_summary["total_orders"], np.nan)

print("\n==================== New vs Existing Customer Comparison ====================")
display(status_summary)

plt.figure(figsize=(7, 5))
plt.bar(status_summary["customer_status"], status_summary["session_conversion_rate"])
plt.title("Session Conversion Rate by Customer Status")
plt.xlabel("Customer Status")
plt.ylabel("Session Conversion Rate")
plt.show()

plt.figure(figsize=(7, 5))
plt.bar(status_summary["customer_status"], status_summary["AOV"])
plt.title("AOV by Customer Status")
plt.xlabel("Customer Status")
plt.ylabel("AOV")
plt.show()

if "loyalty_tier" in cdf2.columns:
    tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
    loyalty_summary = (
        cdf2.groupby("loyalty_tier", dropna=False)
        .agg(
            total_customers=("customer_id", "count"),
            total_sessions=("total_sessions", "sum"),
            converted_sessions=("converted_sessions", "sum"),
            total_orders=("total_orders", "sum"),
            net_revenue_total=("net_revenue_total", "sum"),
            refunded_orders=("refunded_orders", "sum"),
            premium_order_share=("premium_order_share", "mean"),
            avg_discount_sensitivity=("discount_sensitivity_score", "mean")
        )
        .reset_index()
    )
    loyalty_summary["session_conversion_rate"] = np.where(loyalty_summary["total_sessions"] > 0, loyalty_summary["converted_sessions"] / loyalty_summary["total_sessions"], np.nan)
    loyalty_summary["AOV"] = np.where(loyalty_summary["total_orders"] > 0, loyalty_summary["net_revenue_total"] / loyalty_summary["total_orders"], np.nan)
    loyalty_summary["refund_rate"] = np.where(loyalty_summary["total_orders"] > 0, loyalty_summary["refunded_orders"] / loyalty_summary["total_orders"], np.nan)
    loyalty_summary["loyalty_tier"] = pd.Categorical(loyalty_summary["loyalty_tier"], categories=tier_order, ordered=True)
    loyalty_summary = loyalty_summary.sort_values("loyalty_tier")

    print("\n==================== Loyalty Tier Comparison ====================")
    display(loyalty_summary)

    plt.figure(figsize=(8, 5))
    plt.bar(loyalty_summary["loyalty_tier"].astype(str), loyalty_summary["session_conversion_rate"])
    plt.title("Session Conversion Rate by Loyalty Tier")
    plt.xlabel("Loyalty Tier")
    plt.ylabel("Session Conversion Rate")
    plt.show()

problem_customers = cdf2[(cdf2["total_add_to_cart"] > 0) & (cdf2["total_orders"] == 0)].copy() if ("total_add_to_cart" in cdf2.columns and "total_orders" in cdf2.columns) else cdf2.head(0).copy()
print("\n==================== Customers Who Added to Cart but Did Not Purchase ====================")
print(len(problem_customers))

for col in ["loyalty_tier", "preferred_device_type", "preferred_traffic_source", "favorite_category"]:
    if col in problem_customers.columns:
        tmp = problem_customers.groupby(col, dropna=False).agg(problem_customers=("customer_id", "count")).reset_index().sort_values("problem_customers", ascending=False)
        print(f"\nProblem segment by {col}")
        display(tmp.head(10))


In [ ]:
# ================================================================
# STEP 4. Category diagnostics and first-purchase cohort
# ================================================================

em = event_master.copy()
pm = purchase_master.copy()
cdf = customer_df.copy()

if "category" in em.columns:
    cat_event = (
        em[em["category"].notna()]
        .groupby("category", dropna=False)
        .agg(
            views=("is_view", "sum"),
            clicks=("is_click", "sum"),
            carts=("is_add_to_cart", "sum"),
            purchase_events=("is_purchase", "sum")
        )
        .reset_index()
    )

    cat_purchase_base = pm[(pm["matched_tx_flag"] == 1) & (pm["category"].notna())].copy() if ("matched_tx_flag" in pm.columns and "category" in pm.columns) else pm.head(0).copy()
    if len(cat_purchase_base) > 0:
        if "is_premium" in cat_purchase_base.columns:
            cat_purchase = (
                cat_purchase_base.groupby("category", dropna=False)
                .agg(
                    orders=("matched_tx_flag", "sum"),
                    net_revenue=("net_revenue", "sum"),
                    refunded_orders=("refund_flag_filled", "sum"),
                    AOV=("net_revenue", "mean"),
                    premium_share=("is_premium", lambda s: pd.Series(s).fillna(0).astype(float).mean())
                )
                .reset_index()
            )
        else:
            cat_purchase = (
                cat_purchase_base.groupby("category", dropna=False)
                .agg(
                    orders=("matched_tx_flag", "sum"),
                    net_revenue=("net_revenue", "sum"),
                    refunded_orders=("refund_flag_filled", "sum"),
                    AOV=("net_revenue", "mean")
                )
                .reset_index()
            )
            cat_purchase["premium_share"] = np.nan
    else:
        cat_purchase = pd.DataFrame(columns=["category", "orders", "net_revenue", "refunded_orders", "AOV", "premium_share"])

    category_perf = cat_event.merge(cat_purchase, on="category", how="left")
    category_perf["view_to_click_rate"] = np.where(category_perf["views"] > 0, category_perf["clicks"] / category_perf["views"], np.nan)
    category_perf["click_to_cart_rate"] = np.where(category_perf["clicks"] > 0, category_perf["carts"] / category_perf["clicks"], np.nan)
    category_perf["cart_to_purchase_rate"] = np.where(category_perf["carts"] > 0, category_perf["purchase_events"] / category_perf["carts"], np.nan)
    category_perf["refund_rate"] = np.where(category_perf["orders"] > 0, category_perf["refunded_orders"] / category_perf["orders"], np.nan)

    print("\n==================== Category Performance Diagnostics ====================")
    display(category_perf.sort_values("orders", ascending=False))

    plt.figure(figsize=(9, 5))
    tmp = category_perf.sort_values("cart_to_purchase_rate", ascending=False)
    plt.bar(tmp["category"], tmp["cart_to_purchase_rate"])
    plt.title("Cart-to-Purchase Rate by Category")
    plt.xlabel("Category")
    plt.ylabel("Cart-to-Purchase Rate")
    plt.show()

valid_orders = pm[pm["matched_tx_flag"] == 1].copy() if "matched_tx_flag" in pm.columns else pm.copy()
valid_orders = valid_orders[valid_orders["timestamp"].notna()].copy() if "timestamp" in valid_orders.columns else valid_orders.head(0).copy()

if len(valid_orders) > 0 and "customer_id" in valid_orders.columns:
    valid_orders["order_month"] = valid_orders["timestamp"].dt.to_period("M").astype(str)
    first_purchase = valid_orders.groupby("customer_id", as_index=False).agg(first_purchase_ts=("timestamp", "min"))
    first_purchase["first_purchase_month"] = first_purchase["first_purchase_ts"].dt.to_period("M").astype(str)
    cohort_df = cdf[["customer_id", "total_orders", "net_revenue_total"]].merge(first_purchase[["customer_id", "first_purchase_month"]], on="customer_id", how="inner")
    cohort_summary = (
        cohort_df.groupby("first_purchase_month", as_index=False)
        .agg(
            total_customers=("customer_id", "count"),
            repeat_buyers=("total_orders", lambda s: (s >= 2).sum()),
            avg_orders_per_customer=("total_orders", "mean"),
            avg_net_revenue_per_customer=("net_revenue_total", "mean")
        )
    )
    cohort_summary["repeat_purchase_rate"] = cohort_summary["repeat_buyers"] / cohort_summary["total_customers"]

    print("\n==================== first_purchase cohort ====================")
    display(cohort_summary)

    plt.figure(figsize=(10, 5))
    plt.plot(cohort_summary["first_purchase_month"], cohort_summary["repeat_purchase_rate"], marker="o")
    plt.title("Repeat Purchase Rate by First Purchase Cohort")
    plt.xlabel("First Purchase Cohort Month")
    plt.ylabel("Repeat Purchase Rate")
    plt.xticks(rotation=45)
    plt.show()
